# BETO RF/RNF classifier
Fine-tuning Spanish BETO. Labels: `0 = RF`, `1 = RNF`.

In [ ]:
!pip -q install transformers datasets evaluate accelerate scikit-learn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from transformers import (AutoTokenizer, BertForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

# Upload/copy ETL outputs into this Drive directory before running.
DATA_DIR = Path('/content/drive/MyDrive/Protoype-Elbeto/training')
MODEL_DIR = Path('/content/drive/MyDrive/Protoype-Elbeto/models/beto_rf_rnf')
MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LENGTH = 128
set_seed(42)
assert all((DATA_DIR / f'{name}.csv').exists() for name in ('train', 'val', 'test')), 'Run ETL and upload CSV files first.'

In [ ]:
def load_split(name):
    frame = pd.read_csv(DATA_DIR / f'{name}.csv', usecols=['text', 'label'])
    frame['label'] = frame['label'].astype('int64')
    return Dataset.from_pandas(frame, preserve_index=False)

dataset = {name: load_split(name) for name in ('train', 'val', 'test')}
dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

tokenized = {name: split.map(tokenize, batched=True, remove_columns=['text'])
            for name, split in dataset.items()}
collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
# BETO has 12 encoder layers: freeze embeddings + first 9, train last 3 + head.
for parameter in model.bert.embeddings.parameters():
    parameter.requires_grad = False
for layer in model.bert.encoder.layer[:9]:
    for parameter in layer.parameters():
        parameter.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,}/{total:,}')

In [ ]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        # RNF (label 1) is positive for academic binary reporting.
        'precision_binary_rnf': precision_score(labels, predictions, pos_label=1, average='binary', zero_division=0),
        'recall_binary_rnf': recall_score(labels, predictions, pos_label=1, average='binary', zero_division=0),
        'f1_binary_rnf': f1_score(labels, predictions, pos_label=1, average='binary', zero_division=0),
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0),
        'f1_weighted': f1_score(labels, predictions, average='weighted', zero_division=0),
    }

In [ ]:
args = TrainingArguments(
    output_dir='/content/beto-checkpoints',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_binary_rnf',
    greater_is_better=True,
    logging_strategy='epoch',
    report_to='none',
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'],
                  eval_dataset=tokenized['val'], processing_class=tokenizer,
                  data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
test_result = trainer.predict(tokenized['test'])
test_metrics = compute_metrics((test_result.predictions, test_result.label_ids))
predictions = np.argmax(test_result.predictions, axis=-1)
print(json.dumps(test_metrics, indent=2))
print(classification_report(test_result.label_ids, predictions, target_names=['RF', 'RNF'], zero_division=0))
print('Confusion matrix [rows=true, columns=predicted]:\n', confusion_matrix(test_result.label_ids, predictions))

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
(MODEL_DIR / 'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
# Zip artifact for faster download from Google Drive.
!cd /content/drive/MyDrive/Protoype-Elbeto && zip -qr beto_rf_rnf.zip models/beto_rf_rnf
print(f'Saved model and archive: {MODEL_DIR} and /content/drive/MyDrive/Protoype-Elbeto/beto_rf_rnf.zip')